In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. 모델 로드 및 분류기(Classifier) 교체
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
print(model)

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Linear(256, 3)
)

# 2. [선택적 미세조정] 전체 동결 후 특정 위치만 동결 해제
# Step 2-1: 전체 레이어 가중치 고정
for param in model.parameters():
    param.requires_grad = False

# Step 2-2: 미세조정을 적용할 특정 위치(layer4 및 fc)만 동결 해제
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

# 3. 동결 해제된(requires_grad=True) 위치의 파라미터만 옵티마이저에 등록
optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},  # 지정 위치 레이어 (낮은 학습률)
    {'params': model.fc.parameters(),     'lr': 1e-3}   # 분류기 레이어 (상대적으로 높은 학습률)
])

criterion = nn.CrossEntropyLoss()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [ ]:
# 4. 가상 데이터로 역전파(Backward) 검증
inputs = torch.randn(1, 3, 224, 224)
labels = torch.tensor([1])

optimizer.zero_grad() # grad 초기화 (순전파 초기화) : 역전파 손실값(경사하강법으로 구한 weight 값)이 저장되어 있음
outputs = model(inputs)
loss = criterion(outputs, labels)
loss.backward()

# 5. 레이어 위치별 Gradient 계산 여부 확인
print("==================================================")
print(" [지정 위치 선택적 미세조정 검증 결과]")
print("==================================================")
print(f"1. 동결 영역 (conv1) Grad 존재 여부 : {model.conv1.weight.grad is not None}")
print(f"2. 해제 영역 (layer4) Grad 존재 여부: {model.layer4[0].conv1.weight.grad is not None}")
print(f"3. 해제 영역 (fc) Grad 존재 여부    : {model.fc[0].weight.grad is not None}")
print("==================================================")

 [지정 위치 선택적 미세조정 검증 결과]
1. 동결 영역 (conv1) Grad 존재 여부 : False
2. 해제 영역 (layer4) Grad 존재 여부: True
3. 해제 영역 (fc) Grad 존재 여부    : True
